### Import Dependencies

In [ ]:
from pydantic import BaseModel

from langsmith import traceable, get_current_run_tree

import instructor

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send

from langchain_core.messages import SystemMessage
from IPython.display import Image, display

from typing import Literal, Dict, Any, Annotated, List
from operator import add

import random
import openai
import pandas as pd

from jinja2 import Template

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    FusionQuery,
)


In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
from typing import TypedDict
from api.agents.utils.prompt_management import prompt_template_config
import cohere
from pydantic import Field


MODEL_NAME_EMBEDDING = "text-embedding-3-small"


class RAGUsedContextSimple(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(
        description="Description of the item corresponding to the id"
    )


class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContextSimple] = Field(
        description="List of items used to answer the question"
    )


def get_embedding(text, model=MODEL_NAME_EMBEDDING):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

In [ ]:

RetrievedData = TypedDict(
    "RetrievedData",
    {
        "retrieved_context_ids": list[str],
        "retrieved_context": list[str],
        "similarity_scores": list[float],
        "retrieved_context_ratings": list[float],
    },
)

HYBRID_SEARCH_COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"


def retrieve_data(
    query, qdrant_client: QdrantClient, k=5, hybrid=True
) -> RetrievedData:
    query_embedding = get_embedding(query)
    if hybrid:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using="text-embedding-3-small",
                    limit=20,
                ),
                Prefetch(
                    query=Document(
                        text=query,
                        model="qdrant/bm25",
                    ),
                    using="bm25",
                    limit=20,
                ),
            ],
            query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])),
            limit=k,
        )
    else:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            query=query_embedding,
            using="text-embedding-3-small",
            limit=k,
        )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }




In [ ]:
def process_context(context: RetrievedData) -> str:
    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


In [ ]:
query = "Can I get a table?"


In [ ]:
answer = retrieve_data(query, qdrant_client, k=10)

In [ ]:
answer


### Multi-intent Questions

In [ ]:
query = "Can I get a tablet for my kid, a watch for me and a laptop for my wife?"


In [ ]:
retrieved_data = retrieve_data(query, qdrant_client, k=10)

In [ ]:
retrieved_data

In [ ]:
class QueryExpandResponse(BaseModel):
  statements: List[str]

In [ ]:
from openai.types.responses.response import Response

MODEL_NAME_EXPANSION = "gpt-5.4-mini"
PROVIDER_NAME_EXPANSION = "openai"

class QueryExpansionNodeOutput(TypedDict):
  queries: List[str]

def query_expansion_node(query: str) -> QueryExpansionNodeOutput:

    prompt_template = """You are a query expansion module in a shopping assistant. Your job is to rewrite a customer's query into distinct statements for semantic product search.

## Instructions

- Expand the question into 1-5 concise statements.
- Each statement should capture a separate product or attribute from the query.
- Use natural product-description language.
- Do not produce multiple statements that express the same intent.

## Examples

Question: "Can I get earphones for me and a waterproof speaker?"
Statements:
- "Personal earphones"
- "Waterproof speaker"

Question: "I need a warm winter jacket for hiking"
Statements:
- "Insulated winter jacket"
- "Hiking outerwear for cold weather"

Question: "Do you have any toys?"
Statements:
- "Toys"

<question>
{{ query }}
</question>
"""

    template = Template(prompt_template)

    prompt = template.render(query=query)

    client = instructor.from_provider(
        PROVIDER_NAME_EXPANSION + "/" + MODEL_NAME_EXPANSION, mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": prompt}],
        reasoning={"effort": "none"},
        response_model=QueryExpandResponse,
    )
    if not isinstance(raw_response, Response):
      raise TypeError("Raw response is not a Response object")

    return {"queries": response.statements}


In [ ]:
answer = query_expansion_node(query)

In [ ]:
answer

## LangGraph

### Query Expansion (Sequential Execution)

In [ ]:
class State(BaseModel):
  expanded_query: List[str] = []
  retrieved_context: Annotated[List[str], add] = []
  initial_query: str = ""
  answer: str = ""

### Query Expansion / Rewriting Node

In [ ]:
from openai.types.responses.response import Response


class QueryExpansionNodeOutput(TypedDict):
    expanded_query: List[str]

### NOTE: Super useful technique for keeping type safety between partial "state" dicts and the actual State class
def _check_query_expansion_output(o: QueryExpansionNodeOutput) -> State:
    return State(**o)  # type error here if any key name or type disagrees with State


@traceable(
  name="query_expansion",
  run_type="llm",
  metadata={
    "ls_provider": PROVIDER_NAME_EXPANSION,
    "ls_model_name": MODEL_NAME_EXPANSION,
  }
)
def query_expansion_node(state: State) -> QueryExpansionNodeOutput:

    prompt_template = """You are a query expansion module in a shopping assistant. Your job is to rewrite a customer's query into distinct statements for semantic product search.

## Instructions

- Expand the question into 1-5 concise statements.
- Each statement should capture a separate product or attribute from the query.
- Use natural product-description language.
- Do not produce multiple statements that express the same intent.

## Examples

Question: "Can I get earphones for me and a waterproof speaker?"
Statements:
- "Personal earphones"
- "Waterproof speaker"

Question: "I need a warm winter jacket for hiking"
Statements:
- "Insulated winter jacket"
- "Hiking outerwear for cold weather"

Question: "Do you have any toys?"
Statements:
- "Toys"

<question>
{{ query }}
</question>
"""

    template = Template(prompt_template)

    prompt = template.render(query=state.initial_query)

    client = instructor.from_provider(
        PROVIDER_NAME_EXPANSION + "/" + MODEL_NAME_EXPANSION, mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": prompt}],
        reasoning={"effort": "none"},
        response_model=QueryExpandResponse,
    )
    if not isinstance(raw_response, Response):
        raise TypeError("Raw response is not a Response object")

    return {"expanded_query": response.statements}


### Retriever Node

In [ ]:
PROVIDER_NAME_EMBEDDING = "openai"

@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": PROVIDER_NAME_EMBEDDING, "ls_model_name": MODEL_NAME_EMBEDDING},
)
def get_embedding(text, model=MODEL_NAME_EMBEDDING):
    response = openai.embeddings.create(input=text, model=model)
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }
    return response.data[0].embedding


RetrievedData = TypedDict(
    "RetrievedData",
    {
        "retrieved_context_ids": list[str],
        "retrieved_context": list[str],
        "similarity_scores": list[float],
        "retrieved_context_ratings": list[float],
    },
)

HYBRID_SEARCH_COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"


@traceable(name="retrieve_data", run_type="retriever")
def retrieve_data(
    query, qdrant_client: QdrantClient, k=5, hybrid=True
) -> str:
    query_embedding = get_embedding(query)
    if hybrid:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using=MODEL_NAME_EMBEDDING,
                    limit=20,
                ),
                Prefetch(
                    query=Document(
                        text=query,
                        model="qdrant/bm25",
                    ),
                    using="bm25",
                    limit=20,
                ),
            ],
            query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])),
            limit=k,
        )
    else:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            query=query_embedding,
            using="text-embedding-3-small",
            limit=k,
        )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    formatted_context = ""

    for id, chunk, rating in zip(
        retrieved_context_ids,
        retrieved_context,
        retrieved_context_ratings,
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

class RetrieverNodeOutput(TypedDict):
  retrieved_context: list[str]

def _check_retriever_output(o: RetrieverNodeOutput) -> State:
    return State(**o)

@traceable(name="retriever_node", run_type="retriever")
def retriever_node(state: State) -> RetrieverNodeOutput:
    qdrant_client = QdrantClient(url="http://localhost:6333")
    retrieved_context: list[str] = []
    for query in state.expanded_query:
        retrieved_context.append(retrieve_data(query, qdrant_client, k=5))

    return {"retrieved_context": retrieved_context}


### Aggregator Node

In [ ]:
class AggregatorNodeOutput(BaseModel):
  answer: str


### NOTE: Super useful technique for keeping type safety between partial "state" dicts and the actual State class
def _check_aggregator_output(o: AggregatorNodeOutput) -> State:
    return State(answer=o.answer)  # type error here if any key name or type disagrees with State


In [ ]:
MODEL_NAME_ANSWER_GEN = "gpt-5.4-mini"
PROVIDER_NAME_ANSWER_GEN = "openai"

@traceable(
  name="generate_answer",
  run_type="llm",
  metadata={
    "ls_provider": PROVIDER_NAME_EXPANSION,
    "ls_model_name": MODEL_NAME_EXPANSION,
  }
)
def aggregator_node(state: State) -> AggregatorNodeOutput:

    preprocessed_context = "\n".join(state.retrieved_context)

    prompt_template = """You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

### Instructions

- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

### Context

{{ preprocessed_context }}

### Question

{{ question }}
"""

    template = Template(prompt_template)

    prompt = template.render(
        preprocessed_context=preprocessed_context, question=state.initial_query
    )

    client = instructor.from_provider(
        "openai/gpt-5.4-mini", mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": prompt}],
        reasoning={"effort": "none"},
        response_model=AggregatorNodeOutput,
    )

    return response

In [ ]:
from enum import StrEnum

class Nodes(StrEnum):
    QUERY_EXPANSION = "query_expansion_node"
    RETRIEVER = "retriever_node"
    AGGREGATOR = "aggregator_node"


workflow = StateGraph(State)

workflow.add_node(Nodes.QUERY_EXPANSION, query_expansion_node)
workflow.add_node(Nodes.RETRIEVER, retriever_node)
workflow.add_node(Nodes.AGGREGATOR, aggregator_node)

workflow.add_edge(START, Nodes.QUERY_EXPANSION)
workflow.add_edge(Nodes.QUERY_EXPANSION, Nodes.RETRIEVER)
workflow.add_edge(Nodes.RETRIEVER, Nodes.AGGREGATOR)
workflow.add_edge(Nodes.AGGREGATOR, END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
query = "Can I get a tablet for my kid, a watch for me and a laptop for my wife?"


In [ ]:
initial_state = State(
  initial_query=query,
)


In [ ]:
result = graph.invoke(initial_state)
result



In [ ]:
result["answer"]

### Query Expansion (Parallel Execution)

In [ ]:
class State(BaseModel):
  expanded_query: List[str] = []
  retrieved_context: Annotated[List[str], add] = []
  initial_query: str = ""
  answer: str = ""
  query: str = ""
  k: int = 10

### Query Expansion / Rewriting Node

In [ ]:
class QueryExpansionNodeOutput(TypedDict):
    expanded_query: List[str]

### NOTE: Super useful technique for keeping type safety between partial "state" dicts and the actual State class
def _check_query_expansion_output(o: QueryExpansionNodeOutput) -> State:
    return State(**o)  # type error here if any key name or type disagrees with State


@traceable(
  name="query_expansion",
  run_type="llm",
  metadata={
    "ls_provider": PROVIDER_NAME_EXPANSION,
    "ls_model_name": MODEL_NAME_EXPANSION,
  }
)
def query_expansion_node(state: State) -> QueryExpansionNodeOutput:

    prompt_template = """You are a query expansion module in a shopping assistant. Your job is to rewrite a customer's query into distinct statements for semantic product search.

## Instructions

- Expand the question into 1-5 concise statements.
- Each statement should capture a separate product or attribute from the query.
- Use natural product-description language.
- Do not produce multiple statements that express the same intent.

## Examples

Question: "Can I get earphones for me and a waterproof speaker?"
Statements:
- "Personal earphones"
- "Waterproof speaker"

Question: "I need a warm winter jacket for hiking"
Statements:
- "Insulated winter jacket"
- "Hiking outerwear for cold weather"

Question: "Do you have any toys?"
Statements:
- "Toys"

<question>
{{ query }}
</question>
"""

    template = Template(prompt_template)

    prompt = template.render(query=state.initial_query)

    client = instructor.from_provider(
        PROVIDER_NAME_EXPANSION + "/" + MODEL_NAME_EXPANSION, mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": prompt}],
        reasoning={"effort": "none"},
        response_model=QueryExpandResponse,
    )
    if not isinstance(raw_response, Response):
        raise TypeError("Raw response is not a Response object")

    return {"expanded_query": response.statements}


In [ ]:
class StateToSend(TypedDict):
  query: str
  k: int

def _check_query_expand_conditional_edges(sts: StateToSend) -> State:
  return State(**sts)

def query_expand_conditional_edges(state: State) -> list[Send]:
  send_messages: list[Send] = []
  for query in state.expanded_query:
    send_messages.append(Send(Nodes.RETRIEVER, StateToSend(query=query, k=state.k)))
  return send_messages

### Retriever Node

In [ ]:
from api.api.models import ItemPayload


PROVIDER_NAME_EMBEDDING = "openai"

@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": PROVIDER_NAME_EMBEDDING, "ls_model_name": MODEL_NAME_EMBEDDING},
)
def get_embedding(text, model=MODEL_NAME_EMBEDDING):
    response = openai.embeddings.create(input=text, model=model)
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }
    return response.data[0].embedding


RetrievedData = TypedDict(
    "RetrievedData",
    {
        "retrieved_context_ids": list[str],
        "retrieved_context": list[str],
        "similarity_scores": list[float],
        "retrieved_context_ratings": list[float],
    },
)

HYBRID_SEARCH_COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"


class RetrieverNodeOutput(TypedDict):
  retrieved_context: list[str]

def _check_retriever_output(o: RetrieverNodeOutput) -> State:
    return State(**o)

@traceable(name="retriever_node_parallel", run_type="retriever")
def retriever_node_parallel(
    state: StateToSend
) -> RetrieverNodeOutput:
    hybrid = True
    qdrant_client = QdrantClient(url="http://localhost:6333")
    query_embedding = get_embedding(state["query"])
    if hybrid:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using=MODEL_NAME_EMBEDDING,
                    limit=20,
                ),
                Prefetch(
                    query=Document(
                        text=state["query"],
                        model="qdrant/bm25",
                    ),
                    using="bm25",
                    limit=20,
                ),
            ],
            query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])),
            limit=state["k"],
        )
    else:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            query=query_embedding,
            using="text-embedding-3-small",
            limit=state["k"],
        )

    retrieved_context_ids: list[str] = []
    retrieved_context: list[str] = []
    similarity_scores: list[float] = []
    retrieved_context_ratings: list[float] = []


    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        payload_validated = ItemPayload.model_validate(result.payload)
        retrieved_context_ids.append(payload_validated.parent_asin)
        retrieved_context.append(payload_validated.preprocessed_description)
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(payload_validated.average_rating)

    formatted_context = ""

    for id, chunk, rating in zip(
        retrieved_context_ids,
        retrieved_context,
        retrieved_context_ratings,
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return {"retrieved_context": [formatted_context]}


### Aggregator Node

In [ ]:
class AggregatorNodeOutput(BaseModel):
  answer: str


### NOTE: Super useful technique for keeping type safety between partial "state" dicts and the actual State class
def _check_aggregator_output(o: AggregatorNodeOutput) -> State:
    return State(answer=o.answer)  # type error here if any key name or type disagrees with State


In [ ]:
MODEL_NAME_ANSWER_GEN = "gpt-5.4-mini"
PROVIDER_NAME_ANSWER_GEN = "openai"

@traceable(
  name="generate_answer",
  run_type="llm",
  metadata={
    "ls_provider": PROVIDER_NAME_EXPANSION,
    "ls_model_name": MODEL_NAME_EXPANSION,
  }
)
def aggregator_node(state: State) -> AggregatorNodeOutput:

    preprocessed_context = "\n".join(state.retrieved_context)

    prompt_template = """You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

### Instructions

- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

### Context

{{ preprocessed_context }}

### Question

{{ question }}
"""

    template = Template(prompt_template)

    prompt = template.render(
        preprocessed_context=preprocessed_context, question=state.initial_query
    )

    client = instructor.from_provider(
        "openai/gpt-5.4-mini", mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": prompt}],
        reasoning={"effort": "none"},
        response_model=AggregatorNodeOutput,
    )

    return response

In [ ]:
from enum import StrEnum

class Nodes(StrEnum):
    QUERY_EXPANSION = "query_expansion_node"
    RETRIEVER = "retriever_node"
    AGGREGATOR = "aggregator_node"


workflow = StateGraph(State)

workflow.add_node(Nodes.QUERY_EXPANSION, query_expansion_node)
workflow.add_node(Nodes.RETRIEVER, retriever_node_parallel)
workflow.add_node(Nodes.AGGREGATOR, aggregator_node)

workflow.add_conditional_edges(
    Nodes.QUERY_EXPANSION, query_expand_conditional_edges

)

workflow.add_edge(START, Nodes.QUERY_EXPANSION)
workflow.add_edge(Nodes.RETRIEVER, Nodes.AGGREGATOR)

workflow.add_edge(Nodes.AGGREGATOR, END)

graph_parallel = workflow.compile()


In [ ]:
display(Image(graph_parallel.get_graph().draw_mermaid_png()))

In [ ]:
query = "Can I get a tablet for my kid, a watch for me and a laptop for my wife?"


In [ ]:
initial_state = State(
  initial_query=query,
)


In [ ]:
result = graph_parallel.invoke(initial_state)
result


In [ ]:
print(result["answer"])